In [2]:
# FAISS is Meta's library for fast similarity search over dense vectors
# faiss-cpu is sufficient for our dataset size
from google.colab import drive
drive.mount('/content/drive')

!pip install faiss-cpu sentence-transformers -q

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.8 MB/s eta 0:00:00


In [3]:
# Loading the reviews with topic assignments from Day 3
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/yelp_NLP/reviews_with_topics.csv')
print(f"Loaded {len(df):,} reviews")
print(df.columns.tolist())
df.head(3)

Loaded 20,000 reviews
['review_id', 'stars', 'sentiment', 'text', 'clean_text', 'date', 'text_length', 'topic_id', 'topic_label', 'year']


,review_id,stars,sentiment,text,clean_text,date,text_length,topic_id,topic_label,year
0,NupoucezhwIHBX4bPD2YDA,3.0,neutral,"Kitschy bright colors, tiny room with lots of ...","kitschy bright colors, tiny room with lots of ...",2013-09-04 17:46:27,724,-1,Other,2013
1,Nh8_8EkP_4J7x0425RnDFg,1.0,negative,This place is very shady!! I took my laptop in...,this place is very shady!! i took my laptop in...,2013-03-20 17:04:30,754,0,General Dining,2013
2,Vs39nBnd-5yZOIJwkWdGpg,3.0,neutral,Dots Diner is like coming home. Friendly and ...,dots diner is like coming home. friendly and a...,2015-10-23 11:11:42,795,0,General Dining,2015


In [4]:
# Converting every review into a 384-dimensional semantic vector
# This is the most time-consuming cell — expect 10-15 minutes on Colab GPU
# all-MiniLM-L6-v2 is fast and accurate, standard choice for semantic search
from sentence_transformers import SentenceTransformer
import torch

print(f"GPU available: {torch.cuda.is_available()}")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode in batches to avoid memory issues
print("Generating embeddings for all reviews...")
embeddings = embedding_model.encode(
    df['clean_text'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print(f"Embeddings shape: {embeddings.shape}")
# Should be (20000, 384) — 20k reviews, 384 dimensions each

GPU available: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for all reviews...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Embeddings shape: (20000, 384)


In [5]:
print(embeddings)

[[ 0.02420525  0.03183876  0.07098305 ...  0.00311501 -0.07260436
   0.02604511]
 [-0.08300518  0.04513318  0.05943214 ... -0.06521266 -0.00882829
  -0.00509918]
 [ 0.01499098 -0.04745304  0.1000931  ...  0.04613401 -0.0501963
   0.03557161]
 ...
 [-0.02031445 -0.0616922   0.0204566  ... -0.03490024 -0.11612955
   0.04469444]
 [ 0.09048041  0.01101218  0.06755828 ... -0.01423124 -0.01131804
   0.02089818]
 [-0.00147352  0.02488331  0.00326845 ... -0.0203013  -0.1280185
   0.06450494]]


In [6]:
# Normalizing vectors so that dot product equals cosine similarity
# Cosine similarity measures the angle between vectors rather than distance
# Better for text because it is not affected by review length
import faiss

# L2 normalize
faiss.normalize_L2(embeddings)
print("Embeddings normalized")
print(f"Sample embedding norm: {np.linalg.norm(embeddings[0]):.4f}")
# Should be very close to 1.0 after normalization

Embeddings normalized
Sample embedding norm: 1.0000


In [7]:
# Building the FAISS index — a data structure optimized for nearest neighbor search(in memory vector db)
# IndexFlatIP uses inner product (equivalent to cosine similarity after normalization)
# For 20k vectors, flat index is fast enough — no need for approximate search
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"FAISS index built")
print(f"Total vectors in index: {index.ntotal:,}")

FAISS index built
Total vectors in index: 20,000


In [8]:
# Testing search quality with diverse queries before building the full pipeline
# Good test queries cover different topics and specificity levels
def semantic_search(query, top_k=5):
    # Encode the query using the same model
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )
    faiss.normalize_L2(query_embedding)

    # Search for top_k most similar reviews
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            'score': float(score),
            'sentiment': df.iloc[idx]['sentiment'],
            'topic': df.iloc[idx]['topic_label'],
            'stars': df.iloc[idx]['stars'],
            'text': df.iloc[idx]['text'][:300]
        })
    return results

# Test with 5 different queries
test_queries = [
    "rude and unfriendly staff",
    "food took too long to arrive",
    "best burger I have ever eaten",
    "dirty tables and bad hygiene",
    "great value for money"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    results = semantic_search(query, top_k=3)
    for i, r in enumerate(results):
        print(f"  Result {i+1} | Score: {r['score']:.3f} | "
              f"Stars: {r['stars']} | Sentiment: {r['sentiment']}")
        print(f"  Text: {r['text'][:150]}...")
        print()


Query: 'rude and unfriendly staff'
--------------------------------------------------
  Result 1 | Score: 0.677 | Stars: 1.0 | Sentiment: negative
  Text: Rude and unprofessional staff. I called today to find out if they had availability over the Thanksgiving holiday and the staff was short and basically...

  Result 2 | Score: 0.609 | Stars: 1.0 | Sentiment: negative
  Text: Ignorant immature staff they were acting like a bunch of children under five reaching over my husbands shoulder to put my chips on the table walk arou...

  Result 3 | Score: 0.608 | Stars: 3.0 | Sentiment: neutral
  Text: The staff overall was very friendly and super nice but there was a housekeeping lady (seemed like a manager or supervisor) who had a bad attitude.I as...


Query: 'food took too long to arrive'
--------------------------------------------------
  Result 1 | Score: 0.753 | Stars: 2.0 | Sentiment: negative
  Text: Service was extremely slow, we were there over an hour before our meals arrived.  F

In [9]:
# Saving the FAISS index and embeddings so Streamlit app can load them
# without recomputing embeddings every time
import os

save_dir = '/content/drive/MyDrive/yelp_NLP/'

# Save FAISS index
faiss.write_index(index, f'{save_dir}faiss_index.bin')

# Save embeddings as numpy array
np.save(f'{save_dir}embeddings.npy', embeddings)

# Save a lightweight version of the dataframe for the app
# Only columns needed for display
app_df = df[['review_id', 'stars', 'sentiment', 'topic_label',
             'text', 'clean_text', 'date']].copy()
app_df.to_csv(f'{save_dir}app_reviews.csv', index=False)

print("Saved:")
for f in os.listdir(save_dir):
    size = os.path.getsize(f'{save_dir}{f}') / (1024*1024)
    print(f"  {f}: {size:.1f} MB")

Saved:
  yelp_academic_dataset_business.json: 113.4 MB
  yelp_academic_dataset_review.json: 5094.4 MB
  reviews_balanced_50k.csv: 59.8 MB
  distilbert_sentiment: 0.0 MB
  bertopic_model: 0.0 MB
  reviews_with_topics.csv: 24.5 MB
  faiss_index.bin: 29.3 MB
  embeddings.npy: 29.3 MB
  app_reviews.csv: 24.2 MB


In [10]:
# Measuring how fast the search is — important for the Streamlit app
# FAISS is designed to be fast but good to know the actual latency
import time

queries = [
    "cold food and slow service",
    "friendly staff and clean environment",
    "overpriced for the quality",
    "best pizza in the city",
    "rude manager would not refund"
]

times = []
for query in queries:
    start = time.time()
    results = semantic_search(query, top_k=5)
    elapsed = (time.time() - start) * 1000
    times.append(elapsed)
    print(f"Query: '{query[:40]}' -> {elapsed:.1f}ms")

print(f"\nAverage search time: {np.mean(times):.1f}ms")
print(f"This means your Streamlit app can handle search queries in real time")

Query: 'cold food and slow service' -> 13.9ms
Query: 'friendly staff and clean environment' -> 9.8ms
Query: 'overpriced for the quality' -> 9.2ms
Query: 'best pizza in the city' -> 9.0ms
Query: 'rude manager would not refund' -> 8.7ms

Average search time: 10.1ms
This means your Streamlit app can handle search queries in real time
